# Recommendation System - Three Approaches

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt

# Load datasets
content_df = pd.read_csv("content.csv") # destinations dataset
interactions_df = pd.read_csv("interactions.csv") # reviews dataset
users_df = pd.read_excel("users.xlsx") # user history dataset

In [4]:
# دمج interactions مع content
interactions_content = pd.merge(
    interactions_df,
    content_df,
    on='content_id',
    how='inner'
)

# دمج الناتج مع users
df = pd.merge(
    interactions_content,
    users_df,
    on='user_id',
    how='inner'
)

# عرض البيانات
print(df.head())

   user_id  content_id  time_spent  rating_x            timestamp  \
0      239         442         687         2  2024-06-12 06:19:55   
1     2330         684        1999         1  2025-06-28 04:26:12   
2     2888        1010        1057         4  2024-07-14 02:00:35   
3     3107         661         561         2  2025-10-10 17:36:46   
4     3642        1218        1328         3  2025-04-21 13:03:40   

                                     title  \
0                     SQL for Data Science   
1  Natural Language Processing with Python   
2                   Computer Vision Basics   
3               Power BI for Data Analysis   
4                     SQL for Data Science   

                                         description              category  \
0      Build projects and gain practical experience.        Cyber Security   
1  Perfect for beginners looking to start their j...    Business Analytics   
2  Beginner-friendly course covering core concept...    Business Analytics

In [69]:
df.to_csv("df.csv", index=False)
df_ml = pd.read_csv('df.csv')

# توحيد الأعمدة (مهم جدًا)
df_ml['rating'] = df_ml['rating_x']
df_ml['level'] = df_ml['level_x']

## 1) Content-Based Recommendation
**Using: Cosine Similarity**

This approach recommends content based on similarity between items using their features.

In [70]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# إنشاء feature text
content_df['features'] = (
    content_df['category'].astype(str) + ' ' +
    content_df['level'].astype(str) + ' ' +
    content_df['difficulty'].astype(str) + ' ' +
    content_df['description'].astype(str)
)

# TF-IDF
vectorizer = TfidfVectorizer(stop_words='english')

feature_vectors = vectorizer.fit_transform(content_df['features'])

# Cosine Similarity
cosine_sim = cosine_similarity(feature_vectors)

print(cosine_sim.shape)

(2000, 2000)


In [26]:
# Recommendation Function
def recommend_content(user_id, interactions_df, content_df, cosine_sim):

    # الكورسات اللي المستخدم شافها
    visited_content = interactions_df[
        interactions_df['user_id'] == user_id
    ]['content_id'].values

    # لو المستخدم مش موجود
    if len(visited_content) == 0:
        return "No history found for this user"

    # تحويل content_id إلى index
    visited_indices = []

    for content_id in visited_content:
        idx = content_df[
            content_df['content_id'] == content_id
        ].index

        if len(idx) > 0:
            visited_indices.append(idx[0])

    # حساب similarity scores
    similar_scores = np.sum(
        cosine_sim[visited_indices],
        axis=0
    )

    # ترتيب أعلى النتائج
    recommended_indices = np.argsort(similar_scores)[::-1]

    recommendations = []

    for idx in recommended_indices:

        recommended_content_id = content_df.iloc[idx]['content_id']

        # استبعاد الكورسات اللي المستخدم شافها
        if recommended_content_id not in visited_content:

            recommendations.append(
                content_df.iloc[idx][[
                    'content_id',
                    'title',
                    'category',
                    'level',
                    'difficulty',
                    'rating'
                ]]
            )

        # أعلى 5 توصيات
        if len(recommendations) >= 5:
            break

    return pd.DataFrame(recommendations)


# Example
recommended_courses = recommend_content(
    1,
    interactions_df,
    content_df,
    cosine_sim
)

print(recommended_courses['title'])

1691    Artificial Intelligence Fundamentals
1160                Cloud Computing with AWS
667                     Python for Beginners
578               Full Stack Web Development
1138                    Python for Beginners
Name: title, dtype: object


## 2) Collaborative Filtering
**Using: User Similarity**

This approach recommends content by finding similar users and suggesting what they liked.

In [8]:
# إنشاء User-Item Matrix
user_item_matrix = interactions_df.pivot_table(
    index='user_id',
    columns='content_id',
    values='rating',
    fill_value=0
)

# حساب التشابه بين المستخدمين
user_similarity = cosine_similarity(user_item_matrix)

print(user_item_matrix.shape)
print(user_similarity.shape)

(4873, 1970)
(4873, 4873)


In [27]:
# Collaborative Filtering Recommendation Function

def collaborative_recommend(user_id, user_similarity, user_item_matrix, content_df):

    # التأكد إن المستخدم موجود
    if user_id not in user_item_matrix.index:
        return "User not found"

    # الحصول على index الخاص بالمستخدم
    user_index = user_item_matrix.index.get_loc(user_id)

    # المستخدمين المشابهين
    similar_users = user_similarity[user_index]

    # أعلى 5 مستخدمين مشابهين
    similar_users_idx = np.argsort(similar_users)[::-1][1:6]

    # متوسط تقييمات المستخدمين المشابهين
    similar_user_ratings = user_item_matrix.iloc[
        similar_users_idx
    ].mean(axis=0)

    # الكورسات اللي المستخدم شافها بالفعل
    watched_content = user_item_matrix.loc[user_id]

    watched_content_ids = watched_content[
        watched_content > 0
    ].index

    # حذف الكورسات المشاهدة
    similar_user_ratings = similar_user_ratings.drop(
        watched_content_ids,
        errors='ignore'
    )

    # أعلى 5 توصيات
    recommended_content_ids = similar_user_ratings.sort_values(
        ascending=False
    ).head(5).index

    # عرض بيانات الكورسات
    recommendations = content_df[
        content_df['content_id'].isin(recommended_content_ids)
    ][[
        'content_id',
        'title',
        'category',
        'level',
        'difficulty',
        'rating'
    ]]

    return recommendations


# Example
collaborative_recommendations = collaborative_recommend(
    1,
    user_similarity,
    user_item_matrix,
    content_df
)

print(collaborative_recommendations['title'])

1113                    Advanced Excel for Business
1203    Web Development with HTML, CSS & JavaScript
1436                           Python for Beginners
1471                           Machine Learning A-Z
1993                 Introduction to Cyber Security
Name: title, dtype: object


## 3) ML Model - Predict Like Probability
**Using: Classification (RandomForest)**

This approach trains an ML model to predict the probability that a user will like content (binary classification: like/dislike).

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# تحميل الداتا
data = pd.read_csv('df.csv')

# تحضير البيانات
df_ml = data.copy()

# تحديد الأعمدة المستخدمة
features = [
    'age',
    'interest',
    'learning_style',
    'category',
    'level',
    'difficulty',
    'duration'
]

# إنشاء الهدف: liked (1 إذا rating >= 3، 0 غير ذلك)
df_ml['liked'] = (df_ml['rating'] >= 3).astype(int)

# تحويل البيانات النصية لأرقام
label_encoders = {}

for col in features:
    if df_ml[col].dtype == 'object':
        le = LabelEncoder()
        df_ml[col] = le.fit_transform(df_ml[col])
        label_encoders[col] = le
X = df_ml[features]
y = df_ml['liked']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.966

              precision    recall  f1-score   support

           0       0.96      0.87      0.91      7895
           1       0.97      0.99      0.98     32105

    accuracy                           0.97     40000
   macro avg       0.96      0.93      0.94     40000
weighted avg       0.97      0.97      0.97     40000



In [22]:
# حفظ النموذج والـ encoders
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("label_encoders.pkl", "wb") as f:
    pickle.dump(label_encoders, f)

print("Model and encoders saved successfully!")

Model and encoders saved successfully!
